In [3]:
import pandas as pd
import numpy as np

# Create the datasets
df = pd.DataFrame({'key': ['K0', 'K1', 'K2', 'K3', 'K4', 'K5'],
                   'A': ['A0', 'A1', 'A2', 'A3', 'A4', 'A5']})

other = pd.DataFrame({'key': ['K0', 'K1', 'K2'],
                      'B': ['B0', 'B1', 'B2']})

print("Original DataFrames:")
print("df:")
print(df)
print("\nother:")
print(other)



Original DataFrames:
df:
  key   A
0  K0  A0
1  K1  A1
2  K2  A2
3  K3  A3
4  K4  A4
5  K5  A5

other:
  key   B
0  K0  B0
1  K1  B1
2  K2  B2


In [4]:
# Method 1: Using matrix operations with boolean indexing
print("\n1. Matrix-based join using boolean operations:")

# Create a boolean matrix for matching keys
# For each key in df, check if it exists in other
match_matrix = df['key'].values[:, np.newaxis] == other['key'].values[np.newaxis, :]
print(f"Match matrix shape: {match_matrix.shape}")
print("Match matrix:")
print(match_matrix)

# Find the indices where matches occur
df_indices, other_indices = np.where(match_matrix)
print(f"\nMatching indices - df: {df_indices}, other: {other_indices}")

# Create result using matrix indexing
result = df.copy()
print(result)
result['B'] = np.nan  # Initialize with NaN
result.loc[df_indices, 'B'] = other.loc[other_indices, 'B'].values
print("\nResult:")
print(result)




1. Matrix-based join using boolean operations:
Match matrix shape: (6, 3)
Match matrix:
[[ True False False]
 [False  True False]
 [False False  True]
 [False False False]
 [False False False]
 [False False False]]

Matching indices - df: [0 1 2], other: [0 1 2]
  key   A
0  K0  A0
1  K1  A1
2  K2  A2
3  K3  A3
4  K4  A4
5  K5  A5

Result:
  key   A    B
0  K0  A0   B0
1  K1  A1   B1
2  K2  A2   B2
3  K3  A3  NaN
4  K4  A4  NaN
5  K5  A5  NaN


C:\Users\72526\AppData\Local\Temp\ipykernel_31484\1632974076.py:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['B0' 'B1' 'B2']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  result.loc[df_indices, 'B'] = other.loc[other_indices, 'B'].values


In [3]:
# Method 2: Using broadcasting and vectorized operations
print("\n\n2. Using broadcasting for matrix operations:")

# Create a mapping matrix
keys_df = df['key'].values
keys_other = other['key'].values

# Create a boolean mask using broadcasting
mask = keys_df[:, None] == keys_other[None, :]
print("Broadcasting mask:")
print(mask)

# Apply the mask to get values
result2 = df.copy()
result2['B'] = np.where(mask.any(axis=1), 
                       other['B'].values[mask.argmax(axis=1)], 
                       np.nan)
print("\nResult using broadcasting:")
print(result2)





2. Using broadcasting for matrix operations:
Broadcasting mask:
[[ True False False]
 [False  True False]
 [False False  True]
 [False False False]
 [False False False]
 [False False False]]

Result using broadcasting:
  key   A    B
0  K0  A0   B0
1  K1  A1   B1
2  K2  A2   B2
3  K3  A3  NaN
4  K4  A4  NaN
5  K5  A5  NaN


In [ ]:
# Method 3: Matrix multiplication approach for categorical encoding
print("\n\n3. Using matrix multiplication (one-hot encoding approach):")

# Get unique keys from both dataframes
all_keys = pd.concat([df['key'], other['key']]).unique()
print(f"All unique keys: {all_keys}")

# Create one-hot encoded matrices
df_onehot = pd.get_dummies(df['key']).reindex(columns=all_keys, fill_value=0)
other_onehot = pd.get_dummies(other['key']).reindex(columns=all_keys, fill_value=0)

print("\nOne-hot encoded matrices:")
print("df one-hot:")
print(df_onehot)
print("\nother one-hot:")
print(other_onehot)

# Matrix multiplication to find matches
match_result = df_onehot.values @ other_onehot.T.values
print("\nMatrix multiplication result:")
print(match_result)

# Use the multiplication result to map values
result3 = df.copy()
result3['B'] = np.nan
for i, row in enumerate(match_result):
    match_idx = np.where(row > 0)[0]
    if len(match_idx) > 0:
        result3.loc[i, 'B'] = other.loc[match_idx[0], 'B']

print("\nFinal result using matrix multiplication:")
print(result3)





3. Using matrix multiplication (one-hot encoding approach):
All unique keys: ['K0' 'K1' 'K2' 'K3' 'K4' 'K5']

One-hot encoded matrices:
df one-hot:
      K0     K1     K2     K3     K4     K5
0   True  False  False  False  False  False
1  False   True  False  False  False  False
2  False  False   True  False  False  False
3  False  False  False   True  False  False
4  False  False  False  False   True  False
5  False  False  False  False  False   True

other one-hot:
      K0     K1     K2  K3  K4  K5
0   True  False  False   0   0   0
1  False   True  False   0   0   0
2  False  False   True   0   0   0

Matrix multiplication result:
[[1 0 0]
 [0 1 0]
 [0 0 1]
 [0 0 0]
 [0 0 0]
 [0 0 0]]

Final result using matrix multiplication:
  key   A    B
0  K0  A0   B0
1  K1  A1   B1
2  K2  A2   B2
3  K3  A3  NaN
4  K4  A4  NaN
5  K5  A5  NaN


C:\Users\72526\AppData\Local\Temp\ipykernel_28604\139217588.py:29: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'B0' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  result3.loc[i, 'B'] = other.loc[match_idx[0], 'B']


In [5]:
result3

,key,A,B
0,K0,A0,B0
1,K1,A1,B1
2,K2,A2,B2
3,K3,A3,NaN
4,K4,A4,NaN
5,K5,A5,NaN


In [6]:
# Method 4: Pure NumPy matrix approach
print("\n\n4. Pure NumPy matrix operations:")

# Convert to numpy arrays
df_keys = df['key'].values
other_keys = other['key'].values
other_values = other['B'].values

# Create comparison matrix
comparison = df_keys[:, np.newaxis] == other_keys
print("Comparison matrix:")
print(comparison)

# Use matrix operations to get the result
result_values = np.full(len(df_keys), np.nan, dtype=object)
row_matches = np.any(comparison, axis=1)
col_indices = np.argmax(comparison, axis=1)
result_values[row_matches] = other_values[col_indices[row_matches]]

result4 = df.copy()
result4['B'] = result_values
print("\nPure NumPy result:")
print(result4)



4. Pure NumPy matrix operations:
Comparison matrix:
[[ True False False]
 [False  True False]
 [False False  True]
 [False False False]
 [False False False]
 [False False False]]

Pure NumPy result:
  key   A    B
0  K0  A0   B0
1  K1  A1   B1
2  K2  A2   B2
3  K3  A3  NaN
4  K4  A4  NaN
5  K5  A5  NaN
